<a href="https://colab.research.google.com/github/spbromberg/Project-1-Group-5-DS-4002/blob/main/Analysis/IMDb_Reviews_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#IMBD Reviews Data Analysis

In [ ]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
#load dataset

imdb = pd.read_csv("imdb_reviews.csv", nrows=10)
print(imdb)

   review_id                                             review  rating  \
0          0  Once again Mr. Costner has dragged out a movie...       2   
1          1  This is a pale imitation of 'Officer and a Gen...       3   
2          2  It seems ever since 1982, about every two or t...       3   
3          3  Wow, another Kevin Costner hero movie. Postman...       4   
4          4  Alas, another Costner movie that was an hour t...       4   
5          5  I wish I knew what to make of a movie like thi...       4   
6          6  A tough sell: British playwright Ronald Harwoo...       3   
7          7  Blake Edwards' legendary fiasco, begins to see...       1   
8          8  I'm not a big fan of musicals, although this t...       2   
9          9  David Bryce's comments nearby are exceptionall...       4   

  sentiment split  
0  negative  test  
1  negative  test  
2  negative  test  
3  negative  test  
4  negative  test  
5  negative  test  
6  negative  test  
7  negative  t

In [ ]:
#clean dataset

# Load dataset
imdb = pd.read_csv("imdb_reviews.csv")

# Function to clean each review
def clean_review(text):
    text = str(text).lower()                  # Make everything lowercase
    text = re.sub(r"<br\s*/?>", " ", text)   # Remove <br /> tags
    text = re.sub(r"<[^>]+>", " ", text)      # Remove any other HTML tags
    text = re.sub(r"[^a-z\s']", " ", text)    # Keep apostrophes! (was [^a-z\s])
    text = re.sub(r"\s+", " ", text)          # Remove extra spaces
    return text.strip()

# Clean the review column
imdb["review_clean"] = imdb["review"].apply(clean_review)

# Look at original vs. cleaned reviews
print(imdb[["review", "review_clean"]].head())

                                              review  \
0  Once again Mr. Costner has dragged out a movie...   
1  This is a pale imitation of 'Officer and a Gen...   
2  It seems ever since 1982, about every two or t...   
3  Wow, another Kevin Costner hero movie. Postman...   
4  Alas, another Costner movie that was an hour t...   

                                        review_clean  
0  once again mr costner has dragged out a movie ...  
1  this is a pale imitation of officer and a gent...  
2  it seems ever since about every two or three y...  
3  wow another kevin costner hero movie postman t...  
4  alas another costner movie that was an hour to...  


In [ ]:
# Data validation check

print("Data Quality")
print("Total reviews: " + str(len(imdb)))
print("Missing values in review_clean: " + str(imdb['review_clean'].isna().sum()))
print("Duplicate reviews: " + str(imdb['review_clean'].duplicated().sum()))

print("\nClass balance check:")
print(imdb['sentiment'].value_counts())

print("\nTrain/test split check:")
print(imdb['split'].value_counts())

print("\nSample cleaned review:")
print(imdb['review_clean'].iloc[0])
print("Word count: " + str(len(imdb['review_clean'].iloc[0].split())))

In [ ]:
# Create Text Features
# Word Count vs TF-IDF Setup

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
import numpy as np

# Separate training and testing data based on the split column
X_train_text = imdb[imdb['split'] == 'train']['review_clean']
X_test_text = imdb[imdb['split'] == 'test']['review_clean']
y_train = imdb[imdb['split'] == 'train']['sentiment'].map({'positive': 1, 'negative': 0})
y_test = imdb[imdb['split'] == 'test']['sentiment'].map({'positive': 1, 'negative': 0})

print(f"Training set size: {len(X_train_text)}")
print(f"Testing set size: {len(X_test_text)}")
print(f"Class balance in training: {y_train.value_counts()}")

In [ ]:
# Approach 1: Word Count

# CountVectorizer: How many times does each word appear in each review?
# min_df=5: word must appear in at least 5 documents
# max_df=0.8: word must appear in no more than 80% of documents (removes very common words)

count_vectorizer = CountVectorizer(
    max_features=5000,  # Keep only the top 5000 most frequent words
    min_df=5,           # Word must appear in at least 5 reviews
    max_df=0.8,         # Word can appear in at most 80% of reviews
    stop_words='english' # Remove common English words like 'the', 'a', etc.
)

# Fit on training data only (never fit on test data!)
X_train_count = count_vectorizer.fit_transform(X_train_text)
X_test_count = count_vectorizer.transform(X_test_text)

print("Word Count approach - Feature matrix shape: " + {X_train_count.shape})
print("Number of features created: " + {X_train_count.shape[1]})

# Quick example: what are some of the words it found?
feature_names_count = count_vectorizer.get_feature_names_out()
print("Sample words: " + {feature_names_count[:20]})

In [ ]:
# Approach 2: TF-IDF

# TfidfVectorizer: "Term Frequency-Inverse Document Frequency"
# Same parameters as CountVectorizer, but weights words differently
# Words that appear in many reviews get lower weight (less informative)
# Words unique to fewer reviews get higher weight (more informative for classification)

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,  # Keep only the top 5000 most frequent words
    min_df=5,           # Word must appear in at least 5 reviews
    max_df=0.8,         # Word can appear in at most 80% of reviews
    stop_words='english' # Remove common English words
)

# Fit on training data only
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

print(f"TF-IDF approach - Feature matrix shape: {X_train_tfidf.shape}")
print(f"Number of features created: {X_train_tfidf.shape[1]}")

# The words are the same, but the values are weighted differently
feature_names_tfidf = tfidf_vectorizer.get_feature_names_out()
print(f"Sample words: {feature_names_tfidf[:20]}")

In [ ]:
# Compare Word Count vs TF-IDF

# Train a logistic regression model with each approach and compare
# using 5-fold cross-validation on the training data

lr_model = LogisticRegression(max_iter=1000, random_state=42)

# 5-fold CV with Word Count features
cv_scores_count = cross_val_score(
    lr_model, X_train_count, y_train,
    cv=5, scoring='accuracy'
)

# 5-fold CV with TF-IDF features
cv_scores_tfidf = cross_val_score(
    lr_model, X_train_tfidf, y_train,
    cv=5, scoring='accuracy'
)

print("Word Count approach - 5-fold CV Accuracy:")
print(f"  Scores: {cv_scores_count}")
print(f"  Mean: {cv_scores_count.mean():.4f} (+/- {cv_scores_count.std():.4f})")

print("\nTF-IDF approach - 5-fold CV Accuracy:")
print(f"  Scores: {cv_scores_tfidf}")
print(f"  Mean: {cv_scores_tfidf.mean():.4f} (+/- {cv_scores_tfidf.std():.4f})")

# Which one is better?
if cv_scores_tfidf.mean() > cv_scores_count.mean():
    print("\nTF-IDF performs better! We'll use TF-IDF for final model.")
    better_approach = "tfidf"
else:
    print("\nWord Count performs better! We'll use Word Count for final model.")
    better_approach = "count"